# Feature Engineering

## Objective

The objective of this notebook is to construct features that capture behavioural patterns identified during exploratory analysis.

These features aim to enhance the detection of anomalous and fraudulent transactions by encoding structural, statistical, and interaction-based signals.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os 
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("C:/Users/marc/Desktop/UOC/TFM/Code/financial-fraud-anomaly-detection/data/raw/PS_20174392719_1491204439457_log.csv")

# Quick check
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Feature Engineering Strategy

Based on the EDA findings, feature construction focuses on:

- Capturing balance consistency anomalies  
- Encoding transaction type risk  
- Handling skewed distributions  
- Modelling interactions between variables  
- Representing behavioural deviations  

These features aim to move beyond simple statistical relationships and capture structural patterns in transaction behaviour.

### Amount-based Features

Transaction amounts are highly skewed and require transformation to stabilize variance and improve model performance. 

In [9]:
# Log transformation

df['log_amount'] = np.log1p(df['amount'])

# (Check later) Relative amount vs Origin balance
df['amount_to_balance_ratio'] = df['amount']/(df['oldbalanceOrg'] + 1)

### Balance Consistency Features 

These features capture deviations from expected balance updates and represent key behavioural signals for anomaly detection

In [10]:
# Expected balance after transaction
df['expected_newbalanceOrig'] = df['oldbalanceOrg'] - df['amount']

# Error
df['balance_error'] = df['expected_newbalanceOrig'] - df['newbalanceOrig']

# Absolute error
df['abs_balance_error'] = np.abs(df['balance_error'])

# log version
df['log_balance_error'] = np.abs(df['balance_error'])



### Transaction Type Features 

Fraud is structurally concentrated in specific transaction types. 

In [11]:
# One-hot encoding

df = pd.get_dummies(df, columns=['type'], drop_first=True)

# Risk-based flag 
df['high_risk_type'] = df[['type_TRANSFER', 'type_CASH_OUT']].max(axis=1)

### Interaction Features

Fraud behaviour depends on interactions between variables rather than individual features.

In [12]:
# Interaction: amount x type
df['amount_transfer'] = df['amount'] * df.get('type_TRANSFER', 0)
df['amount_cashout'] = df['amount'] * df.get('type_CASH_OUT', 0)

# Interaction: amount x balance error
df['amount_balance_error'] = df['amount'] * df['abs_balance_error']

### Destination Behaviour Features

These features capture patterns related to recipient accounts.

In [13]:
df['dest_balance_change'] = df['newbalanceDest'] - df['oldbalanceDest']

## Feature Cleaning

Handle infinite values, missing values, and scaling issues.

In [15]:
# Check missing values
missing = df.isnull().sum()
missing

step                       0
amount                     0
nameOrig                   0
oldbalanceOrg              0
newbalanceOrig             0
nameDest                   0
oldbalanceDest             0
newbalanceDest             0
isFraud                    0
isFlaggedFraud             0
log_amount                 0
amount_to_balance_ratio    0
expected_newbalanceOrig    0
balance_error              0
abs_balance_error          0
log_balance_error          0
type_CASH_OUT              0
type_DEBIT                 0
type_PAYMENT               0
type_TRANSFER              0
high_risk_type             0
amount_transfer            0
amount_cashout             0
amount_balance_error       0
dest_balance_change        0
dtype: int64

In [19]:
# Check infinite values in numeric features only
numeric_df = df.select_dtypes(include=[np.number])

inf_counts = np.isinf(numeric_df).sum()

# Display only columns with issues
inf_counts

step                       0
amount                     0
oldbalanceOrg              0
newbalanceOrig             0
oldbalanceDest             0
newbalanceDest             0
isFraud                    0
isFlaggedFraud             0
log_amount                 0
amount_to_balance_ratio    0
expected_newbalanceOrig    0
balance_error              0
abs_balance_error          0
log_balance_error          0
amount_transfer            0
amount_cashout             0
amount_balance_error       0
dest_balance_change        0
dtype: int64

In [21]:
# Check for extreme values 
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,amount_to_balance_ratio,expected_newbalanceOrig,balance_error,abs_balance_error,log_balance_error,amount_transfer,amount_cashout,amount_balance_error,dest_balance_change
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06,1.084087e+01,7.067448e+04,6.540212e+05,-2.010925e+05,2.010925e+05,2.010925e+05,7.627235e+04,6.198909e+04,3.917515e+11,1.242947e+05
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03,1.814509e+00,5.084243e+05,2.952326e+06,6.066505e+05,6.066505e+05,6.066505e+05,5.996106e+05,1.337712e+05,1.560114e+13,8.129391e+05
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-9.244552e+07,-9.244552e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.306083e+07
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.502306e+00,2.344011e-01,-1.464904e+05,-2.496411e+05,2.954230e+03,2.954230e+03,0.000000e+00,0.000000e+00,1.196184e+07,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00,1.122355e+01,6.453832e+00,-1.461913e+04,-6.867726e+04,6.867726e+04,6.867726e+04,0.000000e+00,0.000000e+00,4.567133e+09,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00,1.224876e+01,1.228776e+04,4.948488e+04,-2.954230e+03,2.496411e+05,2.496411e+05,0.000000e+00,8.369619e+04,5.047829e+10,1.491054e+05
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00,1.834213e+01,9.244552e+07,4.958504e+07,1.000000e-02,9.244552e+07,9.244552e+07,9.244552e+07,1.000000e+07,8.546174e+15,1.056878e+08


### Numerical Stability Assessment

The descriptive statistics reveal the presence of extreme values in several engineered features:

- Interaction features (e.g., amount_balance_error) show very large magnitudes.
- Ratio features exhibit instability due to division by near-zero balances.
- These issues may negatively impact model performance and require transformation.

### Feature Stabilization

To address the identified numerical issues, the following transformations are applied:

In [22]:
# Avoid division explosion
df['amount_to_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1e-6)

# Log transformations for stability
df['log_amount_balance_error'] = np.log1p(df['amount_balance_error'])
df['log_amount_to_balance_ratio'] = np.log1p(df['amount_to_balance_ratio'])

In [23]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,amount_to_balance_ratio,expected_newbalanceOrig,balance_error,abs_balance_error,log_balance_error,amount_transfer,amount_cashout,amount_balance_error,dest_balance_change,log_amount_balance_error,log_amount_to_balance_ratio
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06,1.084087e+01,7.060226e+10,6.540212e+05,-2.010925e+05,2.010925e+05,2.010925e+05,7.627235e+04,6.198909e+04,3.917515e+11,1.242947e+05,1.789605e+01,9.033167e+00
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03,1.814509e+00,5.084300e+11,2.952326e+06,6.066505e+05,6.066505e+05,6.066505e+05,5.996106e+05,1.337712e+05,1.560114e+13,8.129391e+05,9.609300e+00,1.111099e+01
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-9.244552e+07,-9.244552e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.306083e+07,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.502306e+00,2.344134e-01,-1.464904e+05,-2.496411e+05,2.954230e+03,2.954230e+03,0.000000e+00,0.000000e+00,1.196184e+07,0.000000e+00,1.629723e+01,2.105959e-01
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00,1.122355e+01,6.458505e+00,-1.461913e+04,-6.867726e+04,6.867726e+04,6.867726e+04,0.000000e+00,0.000000e+00,4.567133e+09,0.000000e+00,2.224215e+01,2.009355e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00,1.224876e+01,1.213350e+10,4.948488e+04,-2.954230e+03,2.496411e+05,2.496411e+05,0.000000e+00,8.369619e+04,5.047829e+10,1.491054e+05,2.464481e+01,2.321924e+01
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00,1.834213e+01,9.244552e+13,4.958504e+07,1.000000e-02,9.244552e+07,9.244552e+07,9.244552e+07,1.000000e+07,8.546174e+15,1.056878e+08,3.668426e+01,3.215764e+01


### Final Feature Selection and Stabilization

Based on the numerical assessment, unstable and redundant features are removed to ensure model robustness and prevent dominance of extreme values.

- `amount_balance_error` is removed due to extreme magnitude caused by the multiplication of large values, which can destabilize model training.

- `amount_to_balance_ratio` is removed in its raw form due to numerical instability arising from division by near-zero balances. Its log-transformed version is retained.

- `expected_newbalanceOrig` is removed as it is a deterministic transformation of existing variables (`oldbalanceOrg - amount`) and does not provide additional information.

- `amount_transfer` and `amount_cashout` are removed due to redundancy with transaction type encoding, as they largely replicate information already captured by categorical features.

Log-transformed features are retained as they provide a stable and scalable representation of behavioural patterns.

In [24]:
# Drop unstable raw features
df.drop(columns=[
    'amount_balance_error',
    'amount_to_balance_ratio',
    'expected_newbalanceOrig'
], inplace=True)

# Optional: drop redundant interaction features
df.drop(columns=[
    'amount_transfer',
    'amount_cashout'
], inplace=True)

In [25]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,balance_error,abs_balance_error,log_balance_error,dest_balance_change,log_amount_balance_error,log_amount_to_balance_ratio
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06,1.084087e+01,-2.010925e+05,2.010925e+05,2.010925e+05,1.242947e+05,1.789605e+01,9.033167e+00
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03,1.814509e+00,6.066505e+05,6.066505e+05,6.066505e+05,8.129391e+05,9.609300e+00,1.111099e+01
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-9.244552e+07,0.000000e+00,0.000000e+00,-1.306083e+07,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.502306e+00,-2.496411e+05,2.954230e+03,2.954230e+03,0.000000e+00,1.629723e+01,2.105959e-01
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00,1.122355e+01,-6.867726e+04,6.867726e+04,6.867726e+04,0.000000e+00,2.224215e+01,2.009355e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00,1.224876e+01,-2.954230e+03,2.496411e+05,2.496411e+05,1.491054e+05,2.464481e+01,2.321924e+01
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00,1.834213e+01,1.000000e-02,9.244552e+07,9.244552e+07,1.056878e+08,3.668426e+01,3.215764e+01


### Feature Scaling

Feature scaling is applied to ensure that all numerical variables are on a comparable scale. This is particularly important for distance-based and anomaly detection models, where features with larger magnitudes can disproportionately influence the results.

StandardScaler is used to center features around zero and scale them to unit variance, improving numerical stability and model performance.

In [ ]:
# -------------------------------------------------------------
# Separate target variable
# -------------------------------------------------------------
y = df['isFraud']

# -------------------------------------------------------------
# Remove non-predictive identifier columns if still present
# -------------------------------------------------------------
columns_to_drop = ['isFraud', 'nameOrig', 'nameDest']

# Keep only columns that actually exist in the dataframe
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

X = df.drop(columns=columns_to_drop)

# -------------------------------------------------------------
# Select only numeric feature columns
# -------------------------------------------------------------
X_numeric = X.select_dtypes(include=[np.number])

# -------------------------------------------------------------
# Apply standard scaling to numeric predictors only
# -------------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

# -------------------------------------------------------------
# Rebuild scaled dataframe and add target back
# -------------------------------------------------------------
X_scaled = pd.DataFrame(X_scaled, columns=X_numeric.columns, index=df.index)
df_scaled = pd.concat([X_scaled, y], axis=1)

In [33]:
# Check scaling results
df_scaled.describe().loc[['mean', 'std']]

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,log_amount,balance_error,abs_balance_error,log_balance_error,dest_balance_change,log_amount_balance_error,log_amount_to_balance_ratio,isFraud
mean,0.0,-2.287095e-18,-1.315080e-17,1.257902e-17,9.148379e-18,-6.861284e-18,-7.147171e-19,-2.287095e-17,2.287095e-18,4.574190e-18,4.574190e-18,-2.287095e-18,-1.658144e-17,-3.845178e-17,4.574190e-18
std,1.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


## Feature Summary

The final feature set is designed to capture both statistical properties and behavioural patterns of financial transactions, with a particular focus on anomaly detection.

### 1. Amount-Based Features
- `log_amount`: Log-transformed transaction amount to reduce skewness and stabilize variance.

These features address the highly right-skewed distribution of transaction amounts observed during EDA.

---

### 2. Balance Consistency Features
- `abs_balance_error`: Absolute deviation from expected balance after transaction.
- `log_balance_error`: Log-transformed balance error for numerical stability.

These features capture deviations from expected accounting behaviour and represent key signals of anomalous activity.

---

### 3. Ratio-Based Features
- `log_amount_to_balance_ratio`: Log-transformed ratio between transaction amount and origin account balance.

This feature reflects the relative scale of a transaction with respect to available funds, capturing abnormal transaction patterns.

---

### 4. Interaction Features
- `log_amount_balance_error`: Log-transformed interaction between transaction amount and balance deviation.

This feature models the joint effect of transaction magnitude and behavioural inconsistency, enabling detection of structurally anomalous transactions.

---

### 5. Transaction Type Encoding
- One-hot encoded variables for transaction types (e.g., `type_TRANSFER`, `type_CASH_OUT`, etc.)

These features capture the structural concentration of fraud within specific transaction categories.

---

### 6. Destination Behaviour Features
- `dest_balance_change`: Change in recipient account balance after the transaction.

This feature provides insight into the impact of transactions on destination accounts and may highlight unusual transfer patterns.

---

### Summary of Feature Design

The engineered features aim to:

- Stabilize numerical distributions through logarithmic transformations  
- Capture behavioural inconsistencies in account balance updates  
- Model interactions between transaction characteristics  
- Encode structural patterns linked to transaction types  

Together, these features provide a robust representation of transaction behaviour, enabling the detection of anomalies beyond simple statistical thresholds.

In [ ]:
# Define output path
output_path = "../data/processed/feature_engineered.csv"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save file
df_scaled.to_csv(output_path, index=False)